In [2]:
import numpy as np
k_b = 1.380649*10**(-23) # J/K
Avrogrado = 6.02214076*10**(23) #1/mol
j_mol = 1*10**3 #J/mol
print(f"{j_mol/Avrogrado/k_b} K")


120.27235504272603 K


In [3]:
# LJ-parameters (eps [K], sigma [A]) per element uit het RASPA .def-bestand
ff_path = "../data/force_field/generic_dreiding_uff_mixing_rules.def"
framework_elements = ["O_", "C_", "H_", "Zn_"]

lj_params = {}  # symbool zoals in de CIF -> (eps, sigma)
with open(ff_path) as f:
    for line in f:
        parts = line.split()
        if parts and parts[0] in framework_elements:
            symbol = parts[0].rstrip("_")
            lj_params[symbol] = (float(parts[2]), float(parts[3]))

lj_params["CH4"] = (158.5, 3.72)  
print(lj_params)

# Lorentz-Berthelot: per roosterelement het paar met CH4
def lorentz_berthelot(p1, p2):
    eps_1, sigma_1 = p1
    eps_2, sigma_2 = p2
    return ((eps_1 * eps_2) ** 0.5, (sigma_1 + sigma_2) / 2)

pair_params = {}  # roostersymbool -> (eps_ij, sigma_ij) voor het paar CH4-symbool
for symbol in framework_elements:
    symbol = symbol.rstrip("_")
    pair_params[symbol] = lorentz_berthelot(lj_params["CH4"], lj_params[symbol])
    print(f"CH4-{symbol}: eps = {pair_params[symbol][0]:.3f} K, sigma = {pair_params[symbol][1]:.4f} A")


{'O': (48.1581, 3.03315), 'C': (47.8562, 3.47299), 'H': (7.64893, 2.84642), 'Zn': (62.3992, 2.46155), 'CH4': (158.5, 3.72)}
CH4-O: eps = 87.367 K, sigma = 3.3766 A
CH4-C: eps = 87.093 K, sigma = 3.5965 A
CH4-H: eps = 34.819 K, sigma = 3.2832 A
CH4-Zn: eps = 99.450 K, sigma = 3.0908 A


In [4]:
import ase.io
import ase
import numpy as np
file_path= "/Users/kasperhuysentruyt/Documents/leven /Cusp/cusp-mlff-gcmc/data/IRMOF-1.cif"
atoms = ase.io.read(file_path, format="cif")

def framework_types(file_path):
    atoms = ase.io.read(file_path, format="cif")
    symbols = atoms.get_chemical_symbols()
    return symbols

framework_types(file_path)
symbols = framework_types(file_path)


/Users/kasperhuysentruyt/Documents/leven /Cusp/cusp-mlff-gcmc/.venv/lib/python3.14/site-packages/ase/io/cif.py:411: UserWarning: crystal system 'cubic' is not interpreted for space group Spacegroup(225, setting=1). This may result in wrong setting!
  warnings.warn(


In [ ]:
#distances between CH4 and framework atoms:
positions = atoms.get_positions()
def distances_to_framework(positions, ch4_position,L):
    delta = positions - ch4_position
    distances = np.abs(delta - np.round(delta / L)*L)
    distances = np.linalg.norm(distances, axis=1)    
    return distances

distance_0 = distances_to_framework(positions, np.array([0.0, 0.0, 0.0]), L=atoms.cell[0, 0])
#print(f"distance_0: {distance_0}")
cutoff_distance = 12.0  # Angstrom

#check
if atoms.cell[0, 0]**(1/3)/2 > distance_0.max():
    print(f"Warning: The cutoff distance is larger than half the box length. This may lead to incorrect results.")

eps_i = [lj_params[s][0] for s in symbols if s in lj_params]
sigma_i = [lj_params[s][1] for s in symbols if s in lj_params]
print(f"eps_i: {eps_i}")
print(f"sigma_i: {sigma_i}")
print(np.unique(eps_i, return_counts=True))
print(np.unique(sigma_i, return_counts=True))


lorentz_berthelot_params = [lorentz_berthelot((158.5, 3.72), (eps, sigma)) for eps, sigma in zip(eps_i, sigma_i)]
print(f"lorentz_berthelot_params: {lorentz_berthelot_params}")

L = atoms.cell[0, 0] 
print(f"L: {L}")

eps_i: [62.3992, 62.3992, 62.3992, 62.3992, 62.3992, 62.3992, 62.3992, 62.3992, 62.3992, 62.3992, 62.3992, 62.3992, 62.3992, 62.3992, 62.3992, 62.3992, 62.3992, 62.3992, 62.3992, 62.3992, 62.3992, 62.3992, 62.3992, 62.3992, 62.3992, 62.3992, 62.3992, 62.3992, 62.3992, 62.3992, 62.3992, 62.3992, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48.1581, 48

37.20246068780086


1
1.0
